# Daily Challenge: LangChain Pipelines with Open-Source LLMs

## Fully completed and heavily commented version

This notebook builds a small LangChain application that runs with a local
Hugging Face model.

It contains:

- a CPU-friendly environment;
- a classic `LLMChain` for the required rewrite exercise;
- the modern LCEL equivalent;
- a two-step summary-to-bullets pipeline;
- streaming, iteration, and batch examples;
- a bonus conversation chain with buffer memory;
- measured latency and a short quality analysis.

No paid API or API key is required.

## What you will learn

By the end of the notebook, you will know how to:

1. load a small instruction model from Hugging Face;
2. create a Transformers generation pipeline;
3. wrap that pipeline for LangChain;
4. build a reusable prompt with variables;
5. execute a classic `LLMChain`;
6. compose modern Runnables with the `|` operator;
7. pass the output of one model call into another;
8. retain a short conversation history;
9. measure latency instead of guessing it;
10. identify the limits of a tiny local model.

## General architecture

### Rewrite chain

```text
Source text
    ↓
PromptTemplate
    ↓
FLAN-T5-small
    ↓
Beginner-friendly rewrite
```

### Two-step pipeline

```text
Paragraph
    ↓
Summary prompt → local LLM
    ↓
Intermediate summary
    ↓
Bullet prompt → local LLM
    ↓
Three cleaned bullet points
```

### Conversation chain

```text
Current user message + memory buffer
                ↓
           Conversation prompt
                ↓
             Local LLM
                ↓
       Response + updated memory
```

## Compatibility note

The original student notebook uses imports from an older LangChain release.

This notebook uses:

- `langchain_huggingface.llms.HuggingFacePipeline`;
- `langchain_core.prompts.PromptTemplate`;
- modern LCEL chains such as `prompt | llm | parser`;
- `langchain_classic` only for the specifically requested legacy
  `LLMChain`, `ConversationChain`, and `ConversationBufferMemory`.

The classic components are included to satisfy the exercise. For a new
production project, prefer LCEL and modern chat-history patterns.

# Part 1 — Environment setup

In [ ]:
# Install the packages required by the notebook.
#
# Why use version ranges?
# - They avoid very old LangChain APIs.
# - They keep the notebook inside one major version family.
# - They reduce the risk of incompatible package combinations.
#
# `%pip` is preferred inside Jupyter/Colab because it installs packages
# into the active notebook environment.

%pip install -qU \
    "langchain>=1.0,<2.0" \
    "langchain-core>=1.0,<2.0" \
    "langchain-community>=0.4,<0.5" \
    "langchain-classic>=1.0,<2.0" \
    "langchain-huggingface>=1.0,<2.0" \
    "transformers>=4.45,<5.0" \
    "sentencepiece>=0.2,<1.0" \
    "accelerate>=1.0,<2.0" \
    "pandas>=2.0,<3.0"

After installation, Colab may occasionally request a runtime restart when
an older version of LangChain was already imported. In that case, restart
once and run the notebook again from the beginning.

In [ ]:
# Optional hardware check.
#
# `nvidia-smi` prints NVIDIA GPU information when a GPU is available.
# The shell fallback prints a CPU message when no NVIDIA GPU is detected.
#
# The exercise is designed to work on CPU, so a missing GPU is not an error.

!nvidia-smi || echo "CPU runtime — the notebook can still run"

In [ ]:
# Standard-library imports.
import importlib.metadata as metadata
import re
import statistics
import time
import warnings
from typing import Dict, List

# Third-party libraries.
import pandas as pd
import torch

# Hugging Face classes:
# - AutoTokenizer converts text to model tokens.
# - AutoModelForSeq2SeqLM loads a text-to-text model such as FLAN-T5.
# - pipeline creates a high-level generation interface.
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    pipeline,
)

# Modern LangChain components.
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import (
    RunnableLambda,
    RunnableSequence,
)
from langchain_huggingface.llms import HuggingFacePipeline

# Classic components requested by the exercise.
# These are retained for educational compatibility.
from langchain_classic.chains import ConversationChain, LLMChain
from langchain_classic.memory import ConversationBufferMemory

# Hide expected deprecation warnings from the legacy examples.
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Select the best available device.
# CUDA is used only when a compatible GPU is available.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# The Transformers pipeline expects:
# - 0 for the first CUDA GPU;
# - -1 for CPU.
PIPELINE_DEVICE = 0 if torch.cuda.is_available() else -1

# Store all measured execution times in one dictionary.
timings: Dict[str, float] = {}

print("Selected device:", DEVICE)
print("\nInstalled package versions:")

# Print versions to make experiments reproducible.
for package_name in [
    "langchain",
    "langchain-core",
    "langchain-classic",
    "langchain-huggingface",
    "transformers",
    "torch",
]:
    try:
        print(
            f"- {package_name}: "
            f"{metadata.version(package_name)}"
        )
    except metadata.PackageNotFoundError:
        print(f"- {package_name}: not found")

# Part 2 — Load a tiny open model

We use `google/flan-t5-small`.

It is appropriate for this exercise because it is:

- relatively small;
- instruction-tuned;
- compatible with text-to-text generation;
- able to run on CPU;
- available without a private API key.

Because it is a small model, it may omit details or follow formatting
instructions imperfectly.

In [ ]:
# Hugging Face model identifier.
MODEL_ID = "google/flan-t5-small"

# Start a timer before downloading/loading the model.
load_start = time.perf_counter()

# Load the tokenizer associated with FLAN-T5.
# It converts strings into token IDs and later decodes generated IDs.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load the sequence-to-sequence language model.
# FLAN-T5 reads one text sequence and generates another.
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

# Move the model to the selected CPU or GPU.
model.to(DEVICE)

# Evaluation mode disables training-only behavior such as dropout.
model.eval()

# Save the measured model-loading time.
timings["model_load_seconds"] = (
    time.perf_counter() - load_start
)

# Count the model parameters for documentation.
parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Model:", MODEL_ID)
print("Device:", DEVICE)
print(f"Parameter count: {parameter_count:,}")
print(
    "Model load time:",
    f"{timings['model_load_seconds']:.3f} seconds",
)

In [ ]:
# Create a high-level Hugging Face generation pipeline.
#
# `text2text-generation` is the correct task for T5/FLAN-T5.
# `max_new_tokens` limits the length of the generated answer.
# `do_sample=False` makes generation deterministic/greedy.
# `repetition_penalty` discourages repeated phrases.
# `truncation=True` prevents inputs longer than the model limit from failing.

generation_pipeline = pipeline(
    task="text2text-generation",
    model=model,
    tokenizer=tokenizer,
    device=PIPELINE_DEVICE,
    max_new_tokens=96,
    do_sample=False,
    repetition_penalty=1.05,
    truncation=True,
)

# Wrap the Transformers pipeline as a LangChain LLM Runnable.
llm = HuggingFacePipeline(
    pipeline=generation_pipeline,
)

print("The Hugging Face pipeline and LangChain wrapper are ready.")

## Part 2A — Required classic `LLMChain`

The chain rewrites technical text in language suitable for beginners.

It contains two components:

1. a `PromptTemplate`;
2. the local LLM wrapper.

`LLMChain` is a legacy abstraction, but it is included because the exercise
explicitly asks for it.

In [ ]:
# The prompt contains one variable: {text}.
#
# Clear constraints help a small model:
# - preserve meaning;
# - use simple language;
# - keep the answer short;
# - avoid unsupported facts.

rewrite_template = """
Rewrite the text below for a complete beginner.

Requirements:
- preserve the original meaning;
- use simple English;
- use no more than two short sentences;
- do not add facts.

Text:
{text}

Beginner-friendly rewrite:
""".strip()

# Convert the string template into a LangChain PromptTemplate.
rewrite_prompt = PromptTemplate(
    template=rewrite_template,
    input_variables=["text"],
)

# Build the classic chain requested by the assignment.
# The generated value will be stored under the key "rewrite".
legacy_rewrite_chain = LLMChain(
    prompt=rewrite_prompt,
    llm=llm,
    output_key="rewrite",
)

print("Classic LLMChain created.")

In [ ]:
# Text that will be simplified.
sample_text = (
    "LangChain helps developers build language-model applications by "
    "composing prompts, models, retrievers, tools, and reusable workflows."
)

# Measure only the chain invocation time.
rewrite_start = time.perf_counter()

# `invoke` accepts a dictionary whose keys match the prompt variables.
legacy_rewrite_result = legacy_rewrite_chain.invoke({
    "text": sample_text,
})

timings["legacy_rewrite_seconds"] = (
    time.perf_counter() - rewrite_start
)

# Extract and clean the generated text.
rewritten_text = legacy_rewrite_result["rewrite"].strip()

print("ORIGINAL")
print(sample_text)

print("\nBEGINNER-FRIENDLY REWRITE")
print(rewritten_text)

print(
    "\nMeasured latency:",
    f"{timings['legacy_rewrite_seconds']:.3f} seconds",
)

# Fail early if the model returns an empty output.
assert rewritten_text, "The rewrite chain returned an empty string."

## Part 2B — Modern LCEL equivalent

LCEL means LangChain Expression Language.

The pipe operator connects Runnables:

```python
prompt | llm | output_parser
```

The output parser guarantees that the final result is a normal Python
string.

In [ ]:
# Compose the modern chain.
rewrite_chain = (
    rewrite_prompt
    | llm
    | StrOutputParser()
)

print("Generated Runnable type:", type(rewrite_chain).__name__)
print(
    "Is it a RunnableSequence?",
    isinstance(rewrite_chain, RunnableSequence),
)

In [ ]:
# Test the modern chain with a second sentence.
modern_rewrite_start = time.perf_counter()

modern_rewrite = rewrite_chain.invoke({
    "text": (
        "A prompt template inserts variables into a reusable instruction "
        "before the instruction is sent to a language model."
    )
}).strip()

timings["modern_rewrite_seconds"] = (
    time.perf_counter() - modern_rewrite_start
)

print(modern_rewrite)
print(
    "\nMeasured latency:",
    f"{timings['modern_rewrite_seconds']:.3f} seconds",
)

assert modern_rewrite, "The modern rewrite chain returned no text."

## Test prompt changes quickly

We keep the same source text but change the requested writing style.

This experiment demonstrates that the prompt is part of the program:
changing its wording can change tone, length, and vocabulary.

In [ ]:
# This prompt contains two variables:
# - {style}: the desired writing style;
# - {text}: the source content.
style_prompt = PromptTemplate.from_template(
    """
    Rewrite the following text in a {style} style.
    Preserve the meaning and write one sentence only.

    Text:
    {text}

    Rewrite:
    """.strip()
)

# Compose the prompt, model, and output parser.
style_chain = (
    style_prompt
    | llm
    | StrOutputParser()
)

style_source = (
    "Local open-source models can reduce dependency on external APIs, "
    "but they require enough memory and careful evaluation."
)

# Prepare multiple inputs for a small experiment.
style_inputs = [
    {
        "style": "friendly beginner",
        "text": style_source,
    },
    {
        "style": "concise professional",
        "text": style_source,
    },
    {
        "style": "encouraging teacher",
        "text": style_source,
    },
]

style_results = []

# Run one input at a time so that we can measure each latency.
for item in style_inputs:
    start = time.perf_counter()
    output = style_chain.invoke(item).strip()
    elapsed = time.perf_counter() - start

    style_results.append({
        "style": item["style"],
        "output": output,
        "latency_seconds": elapsed,
    })

display(pd.DataFrame(style_results))

## Stream or iterate over output

`.stream()` exposes LangChain's streaming interface.

With this local Transformers wrapper, the model may emit one completed
chunk instead of one token at a time. The number of emitted chunks is shown
so that this behavior can be observed rather than assumed.

In [ ]:
# Input for the streaming demonstration.
stream_input = {
    "text": (
        "LangChain Runnables can be invoked, batched, streamed, and "
        "combined into larger workflows."
    )
}

streamed_chunks = []

# Iterate over everything emitted by the Runnable.
for chunk in rewrite_chain.stream(stream_input):
    streamed_chunks.append(chunk)
    print(chunk, end="", flush=True)

print("\n\nNumber of emitted chunks:", len(streamed_chunks))

# Part 3 — Two-step pipeline: summarize, then bulletize

The output of the first model call becomes the input of the second.

Keeping the summary as an intermediate value helps us determine whether an
error came from:

- the summarization stage;
- the bullet-generation stage;
- the formatting stage.

In [ ]:
# Prompt used by stage 1.
summary_prompt = PromptTemplate.from_template(
    """
    Summarize the paragraph below in two concise sentences.

    Rules:
    - keep only the main ideas;
    - do not add information;
    - use clear English.

    Paragraph:
    {paragraph}

    Summary:
    """.strip()
)

# Prompt used by stage 2.
bullets_prompt = PromptTemplate.from_template(
    """
    Convert the summary below into exactly three concise bullet points.

    Rules:
    - begin every line with "- ";
    - output exactly three lines;
    - do not add information not present in the summary.

    Summary:
    {summary}

    Three bullet points:
    """.strip()
)

# Stage 1:
# paragraph dictionary → formatted prompt → LLM → string → stripped string.
summary_chain = (
    summary_prompt
    | llm
    | StrOutputParser()
    | RunnableLambda(lambda value: value.strip())
)

# Stage 2:
# summary dictionary → formatted prompt → LLM → string → stripped string.
raw_bullet_chain = (
    bullets_prompt
    | llm
    | StrOutputParser()
    | RunnableLambda(lambda value: value.strip())
)

print("Both stage chains were created.")

## Deterministic formatting helper

A tiny model may return one paragraph instead of three bullet lines.

The helper below:

- removes existing list markers;
- splits the text into candidate points;
- removes duplicates;
- returns exactly three lines.

It does not fabricate new subject-matter facts. When the model generates
fewer than three distinct ideas, the limitation is displayed explicitly.

In [ ]:
def ensure_three_bullets(raw_text: str) -> str:
    # Remove leading and trailing whitespace.
    cleaned = raw_text.strip()

    # First split on line breaks, sentence boundaries, or semicolons.
    fragments = re.split(
        r"(?:\n+|(?<=[.!?])\s+|;\s*)",
        cleaned,
    )

    normalized: List[str] = []

    for fragment in fragments:
        # Remove bullets or numbered-list markers already produced.
        fragment = re.sub(
            r"^\s*(?:[-*•]+|\d+[.)])\s*",
            "",
            fragment,
        ).strip()

        # Keep only non-empty, non-duplicate fragments.
        if fragment and fragment not in normalized:
            normalized.append(fragment)

    # Fallback: try comma-separated or "and"-separated phrases.
    if len(normalized) < 3:
        comma_parts = [
            part.strip()
            for part in re.split(
                r",\s+|\s+and\s+",
                cleaned,
            )
            if part.strip()
        ]

        for part in comma_parts:
            part = re.sub(
                r"^\s*(?:[-*•]+|\d+[.)])\s*",
                "",
                part,
            ).strip()

            if part and part not in normalized:
                normalized.append(part)

            if len(normalized) >= 3:
                break

    # Make a model limitation visible instead of inventing another fact.
    while len(normalized) < 3:
        normalized.append(
            "[No additional distinct point was generated by the model.]"
        )

    # Keep exactly three items and add Markdown bullet markers.
    return "\n".join(
        f"- {item}"
        for item in normalized[:3]
    )


# Wrap the normal Python function as a LangChain Runnable.
bullet_formatter = RunnableLambda(
    ensure_three_bullets
)

In [ ]:
# Build the complete Runnable pipeline.
#
# The mapping {"summary": summary_chain} converts the original
# {"paragraph": ...} input into {"summary": "...generated summary..."}.
#
# That dictionary is passed to the bullet prompt, then the final formatting
# helper guarantees three visible lines.

summarize_then_bullets = (
    {"summary": summary_chain}
    | raw_bullet_chain
    | bullet_formatter
)

print(
    "Complete pipeline type:",
    type(summarize_then_bullets).__name__,
)
print(
    "Is RunnableSequence:",
    isinstance(summarize_then_bullets, RunnableSequence),
)

In [ ]:
# Short paragraph supplied to the two-stage workflow.
paragraph = """
LangChain is a framework for building applications with language models by
composing prompts, models, retrievers, tools, and other components. Its
Runnable interface gives these components common methods such as invoke,
batch, and stream. Developers can connect small reusable stages instead of
writing one large function, which makes experiments and debugging easier.
""".strip()

pipeline_start = time.perf_counter()

# Run stage 1 independently so we can inspect its result.
intermediate_summary = summary_chain.invoke({
    "paragraph": paragraph,
})

# Run stage 2 with the generated summary.
raw_bullets = raw_bullet_chain.invoke({
    "summary": intermediate_summary,
})

# Normalize the formatting without adding hidden facts.
cleaned_bullets = ensure_three_bullets(raw_bullets)

timings["two_step_pipeline_seconds"] = (
    time.perf_counter() - pipeline_start
)

print("ORIGINAL PARAGRAPH")
print(paragraph)

print("\nINTERMEDIATE SUMMARY")
print(intermediate_summary)

print("\nRAW BULLET OUTPUT FROM THE MODEL")
print(raw_bullets)

print("\nCLEANED THREE-BULLET OUTPUT")
print(cleaned_bullets)

print(
    "\nTotal measured latency:",
    f"{timings['two_step_pipeline_seconds']:.3f} seconds",
)

# Confirm that the final output contains exactly three bullet lines.
bullet_lines = [
    line
    for line in cleaned_bullets.splitlines()
    if line.startswith("- ")
]

assert len(bullet_lines) == 3

## Invoke the complete pipeline directly

The previous cell executed each stage separately for debugging.

The next cell calls the composed pipeline in a single invocation.

In [ ]:
direct_start = time.perf_counter()

direct_bullets = summarize_then_bullets.invoke({
    "paragraph": paragraph,
})

timings["direct_pipeline_seconds"] = (
    time.perf_counter() - direct_start
)

print(direct_bullets)

print(
    "\nMeasured direct-pipeline latency:",
    f"{timings['direct_pipeline_seconds']:.3f} seconds",
)

# Part 4 — Bonus conversation chain with memory

The exercise requests `ConversationChain` with a simple buffer memory.

The prompt contains:

- the previous conversation;
- the current user input;
- a concise and encouraging assistant style.

FLAN-T5-small is not primarily a chat model. The memory may be stored
correctly even when the model fails to use it correctly.

In [ ]:
# Define the style and memory placeholders.
conversation_prompt = PromptTemplate(
    template="""
    You are a concise and encouraging teaching assistant.
    Use the previous conversation when it is relevant.
    If you are unsure, say so briefly.

    Previous conversation:
    {history}

    Human:
    {input}

    Assistant:
    """.strip(),
    input_variables=["history", "input"],
)

# Create an in-memory text buffer.
#
# `memory_key` must match {history} in the prompt.
# `input_key` identifies the user's current message.
# `output_key` identifies the chain response.
memory = ConversationBufferMemory(
    memory_key="history",
    input_key="input",
    output_key="response",
    return_messages=False,
)

# Build the classic conversation chain requested by the lesson.
conversation = ConversationChain(
    llm=llm,
    prompt=conversation_prompt,
    memory=memory,
    output_key="response",
    verbose=False,
)

print("Conversation chain created.")

In [ ]:
# Measure two consecutive conversational turns.
conversation_start = time.perf_counter()

# Turn 1 introduces a name and asks a basic question.
first_turn = conversation.invoke({
    "input": (
        "Hi! My name is Maya. In one sentence, what is LangChain?"
    )
})["response"].strip()

# Turn 2 asks the model to reuse information from turn 1.
second_turn = conversation.invoke({
    "input": (
        "What name did I give you, and can LangChain help me "
        "build a chatbot?"
    )
})["response"].strip()

timings["two_conversation_turns_seconds"] = (
    time.perf_counter() - conversation_start
)

print("TURN 1")
print(first_turn)

print("\nTURN 2")
print(second_turn)

# Display the actual memory to distinguish storage from model recall.
print("\nMEMORY BUFFER")
print(memory.buffer)

print(
    "\nTotal two-turn latency:",
    f"{timings['two_conversation_turns_seconds']:.3f} seconds",
)

# Verify that the important user detail was stored.
assert "Maya" in memory.buffer

## How to evaluate the memory result

Check two separate properties:

1. **Storage:** Does the printed buffer contain both user turns and model
   responses?
2. **Use:** Does the second response correctly mention Maya and answer the
   follow-up?

The first property belongs to LangChain memory. The second also depends on
the model's instruction-following ability.

# Observations: latency, quality, and quirks

In [ ]:
# Convert the timing dictionary into a readable table.
timing_table = pd.DataFrame([
    {
        "operation": operation,
        "seconds": seconds,
    }
    for operation, seconds in timings.items()
]).sort_values(
    "seconds",
    ascending=False,
)

display(timing_table.round(3))

# Calculate a median only for generation workflows.
# Model loading is excluded because it measures a different operation.
generation_timings = [
    value
    for key, value in timings.items()
    if key != "model_load_seconds"
]

if generation_timings:
    median_latency = statistics.median(
        generation_timings
    )

    print(
        "Median generation-workflow latency:",
        f"{median_latency:.3f} seconds",
    )

## Written observations

### Latency

The exact values are generated by the timing table. CPU inference is
possible with FLAN-T5-small, although a GPU is normally faster. Initial
model loading includes downloading and initialization, so it may take much
longer than later calls. The two-step workflow performs two generations and
should therefore be slower than a single rewrite.

### Quality

The model generally handles direct rewriting and short summaries, but its
small size can lead to missing details, very short answers, repetition, or
imperfect formatting. The intermediate summary helps identify which stage
introduced an error.

### Quirks

- Prompt changes can strongly affect the answer.
- Greedy generation improves reproducibility but not factual accuracy.
- Local `.stream()` may emit one completed result rather than tokens.
- Memory storage can work even when the model does not recall correctly.
- The formatter guarantees three lines, but cannot create missing knowledge.
- Classic LangChain chains are included for the exercise; LCEL is the more
  composable modern approach.

## Optional batch experiment

A Runnable can process several inputs with `.batch()`.

In [ ]:
# Prepare two independent rewrite requests.
batch_inputs = [
    {
        "text": (
            "A language model predicts text from patterns found "
            "in its training data."
        )
    },
    {
        "text": (
            "A pipeline connects processing stages so that one stage's "
            "output becomes the next stage's input."
        )
    },
]

# LangChain handles the list of dictionaries.
batch_outputs = rewrite_chain.batch(
    batch_inputs
)

# Present the results in a table.
display(pd.DataFrame({
    "input": [
        item["text"]
        for item in batch_inputs
    ],
    "rewrite": [
        output.strip()
        for output in batch_outputs
    ],
}))

# Deliverables checklist

- [x] Packages installed
- [x] CPU/GPU hardware check
- [x] Small Hugging Face model loaded
- [x] Transformers pipeline created
- [x] Pipeline wrapped with `HuggingFacePipeline`
- [x] Tested classic `LLMChain`
- [x] Modern LCEL rewrite chain
- [x] Prompt-style experiments
- [x] Streaming/iteration example
- [x] Two-step summary-to-bullets pipeline
- [x] Intermediate outputs displayed
- [x] Exactly three cleaned bullet lines
- [x] Bonus conversation chain
- [x] Two conversational turns
- [x] Memory buffer displayed
- [x] Measured latency
- [x] Written observations
- [x] Code thoroughly commented

# Conclusion

The central modern LangChain pattern demonstrated here is:

```text
PromptTemplate | HuggingFacePipeline | OutputParser
```

Each component is a Runnable. Small Runnables can be connected into larger
workflows while preserving the ability to invoke, batch, stream, test, and
debug each stage separately.

# References

- LangChain Hugging Face local pipelines:
  https://docs.langchain.com/oss/python/integrations/llms/huggingface_pipelines
- LangChain Python reference:
  https://reference.langchain.com/python/
- FLAN-T5-small:
  https://huggingface.co/google/flan-t5-small
- Hugging Face Transformers pipelines:
  https://huggingface.co/docs/transformers/main_classes/pipelines